In [39]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('customer_support_tickets_work.csv')

# 2. Inspect the first few rows
print("Dataset Head:")
display(df.head())

# 3. Check data types and missing values
print("\nDataset Info:")
df.info()

# 4. Check for duplicates
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

Dataset Head:


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   o

In [40]:
# --- 1. Handle Null Values ---
# For Open/Pending tickets, 'Resolution' and 'Time to Resolution' will naturally be empty.
# We fill categorical nulls with 'No Resolution' and numericals with 0 or NaN where appropriate.

# Fill missing textual fields
df['Resolution'] = df['Resolution'].fillna('No Resolution')
df['Ticket Priority'] = df['Ticket Priority'].fillna('Medium') # Defaulting priority if missing

# Fill missing Satisfaction Ratings with NaN (or 0 if you prefer, but NaN is safer for averages)
# We keep it as float to handle NaNs
df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].astype(float)

# --- 2. Convert Data Types ---
# Convert date columns to datetime objects
date_cols = ['Date of Purchase', 'First Response Time', 'Time to Resolution']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# --- 3. Clean Placeholder Text in Descriptions ---
# The text "{product_purchased}" appears frequently. We can replace it with the actual product name from the 'Product Purchased' column.
def clean_description(row):
    if pd.isna(row['Ticket Description']):
        return ""
    return row['Ticket Description'].replace("{product_purchased}", str(row['Product Purchased']))

df['Ticket Description'] = df.apply(clean_description, axis=1)

# --- 4. Drop Duplicates ---
df.drop_duplicates(inplace=True)

# Verify Cleaning
print("Missing Values after cleaning:\n", df.isnull().sum())

Missing Values after cleaning:
 Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                         0
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64


In [41]:
# --- 1. Calculate Resolution Time ---
# Project PDF Requirement: Resolution_Time = Resolution Date - Date
# Note: Since we don't have a 'Ticket Creation Date', we might use 'First Response Time' 
# or assume 'Date of Purchase' is the reference, but usually, it's (Resolution Time - First Response).
# Let's calculate the duration in days/hours.

df['Resolution_Time_Hours'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds() / 3600

# Handle negative values (if any) or cases where resolution happened before response (data errors)
df['Resolution_Time_Hours'] = df['Resolution_Time_Hours'].apply(lambda x: x if x > 0 else np.nan)

# --- 2. Categorize Satisfaction ---
# Create buckets for satisfaction rating (High, Medium, Low)
def categorize_rating(rating):
    if pd.isna(rating):
        return 'Not Rated'
    elif rating >= 4:
        return 'High'
    elif rating >= 3:
        return 'Medium'
    else:
        return 'Low'

df['Satisfaction_Category'] = df['Customer Satisfaction Rating'].apply(categorize_rating)

# --- 3. Extract Age Groups ---
# Useful for "Category distribution" analysis later
bins = [0, 18, 30, 50, 100]
labels = ['Teen', 'Young Adult', 'Adult', 'Senior']
df['Age_Group'] = pd.cut(df['Customer Age'], bins=bins, labels=labels)

print("New Features Added:")
display(df[['Resolution_Time_Hours', 'Satisfaction_Category', 'Age_Group']].head())

New Features Added:


,Resolution_Time_Hours,Satisfaction_Category,Age_Group
0,NaN,Not Rated,Adult
1,NaN,Not Rated,Adult
2,6.850000,Medium,Adult
3,NaN,Medium,Young Adult
4,19.683333,Low,Senior


In [42]:
# Check the final shape of data
print(f"Final Dataset Shape: {df.shape}")

# Statistical Summary of the new Resolution Time metric
print("\nResolution Time Summary (Hours):")
print(df['Resolution_Time_Hours'].describe())

# Save the cleaned dataset for the next Milestone
df.to_csv('cleaned_customer_support_tickets.csv', index=False)
print("\nFile 'cleaned_customer_support_tickets.csv' saved successfully.")

Final Dataset Shape: (8469, 20)

Resolution Time Summary (Hours):
count    1402.000000
mean        7.588742
std         5.593297
min         0.016667
25%         3.000000
50%         6.375000
75%        11.362500
max        23.466667
Name: Resolution_Time_Hours, dtype: float64

File 'cleaned_customer_support_tickets.csv' saved successfully.
